# All of Us — genetic features: APOL1 and CKD-PRS (PGS002237) (prototype)

**Author:** Odak. Developed on **synthetic** input files; `mock/make_synthetic_genetics.py` generates a small set with the same layout so the notebook runs out of the box. Real Workbench equivalents are listed in `docs/WORKBENCH_INSTRUCTIONS.md` Part B. Keep file layouts and column names as in the synthetic files and the code runs unchanged.

> **If you are an AI assistant running this on the Workbench:** functions and computation logic are fixed. Only point the inputs at the real files, apply the permitted fixes in Part B, and report the numbers.

## Pipeline

1. Inputs: PLINK `.raw` dosage matrix (ACAF threshold callset, PGS002237 variants), PGS Catalog score file (GRCh38 harmonized), `ancestry_preds.tsv`, APOL1 person table
2. APOL1: drop QC-flagged samples, one-hot the genotype, high risk = G1/G1, G1/G2, G2/G2
3. CKD-PRS: parse `.raw` column names (`chr:pos:ref:alt_counted`) → match to the score file on `chr` + `pos` → orient the effect allele → dosage × weight → per-person sum
4. Sanity check: mean PRS by predicted ancestry

Outputs: **`apol1_table`** (person × APOL1 columns), **`dosage_raw[['IID','prs_raw']]`** (person × raw PRS). Join key to the clinical table: `IID` = `person_id`.

## 0. Setup and inputs

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "/content/drive/MyDrive/all_of_us/dummy_sets"   # folder with the (synthetic or real) input files

In [ ]:
dosage_raw = pd.read_csv(DATA_DIR + "/acaf_threshold.dummy.raw.gz", sep = "\t")

beta_scores = pd.read_csv(DATA_DIR + "/PGS002237_hmPOS_GRCh38.dummy.txt.gz", sep = "\t", comment = "#")

ancestry_preds = pd.read_csv(DATA_DIR + "/ancestry_preds.dummy.tsv", sep = "\t")

apol1_raw = pd.read_excel(DATA_DIR + "/aou_genomic_dummy_2026-09-12_VIEWER_ONLY.xlsx",
                            sheet_name = "apol1_person_table")

## 1. APOL1

**QC:** samples with any True in `qc_flag_cols` are dropped (relatedness, sample QC, known issue, low coverage).

**Genotype:** `apol1_genotype` (`G0/G0`, `G0/G1`, `G1/G1`, `G1/G2`, `G2/G2`, …) is one-hot encoded. **High-risk genotype** = two risk alleles: `G1/G1`, `G1/G2`, `G2/G2` → `apol1_high_risk = 1`. One risk allele (G0/G1, G0/G2) is not high risk (recessive model).

In [ ]:
qc_flag_cols = ['related_flag',
       'sample_qc_flag', 'known_issue_8', 'low_coverage_flag']

qc_flags = apol1_raw.loc[:, qc_flag_cols]

qc_failed_idx = qc_flags[qc_flags.any(axis = 1)].index

apol1_qc = apol1_raw.drop(qc_failed_idx)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot = OneHotEncoder(handle_unknown = 'ignore', sparse_output= False)

apol1_one_hot = one_hot.fit_transform(pd.DataFrame(apol1_qc['apol1_genotype']))

apol1_table = pd.concat([apol1_qc, pd.DataFrame(apol1_one_hot, columns = one_hot.get_feature_names_out(), index = apol1_qc.index)], axis = 1)

In [ ]:
apol1_table['apol1_high_risk'] = apol1_table.loc[:,['apol1_genotype_G1/G1',
       'apol1_genotype_G1/G2', 'apol1_genotype_G2/G2']].any(axis = 1).astype(int)

## 2. CKD-PRS — parse the `.raw` column names

In PLINK `--export A` output every variant column is named `chr1:12345:A:G_G`: chromosome, position, reference allele, alternative allele, and after `_` the **counted allele** (the allele whose dosage is 0/1/2). Column names are read from a single person (identical for everyone), split on `:` and `_`, and turned into a table.

In [ ]:
first_person = dosage_raw.loc[dosage_raw['FID'] == dosage_raw.iloc[0]['FID']]

variant_cols = first_person.columns[first_person.columns.str.contains("chr", regex = False, case=False)]

In [ ]:
variant_table = pd.Series(variant_cols).str.split(":", expand = True)
variant_last_field = variant_table.iloc[:,-1].str.split("_", expand = True).rename(columns = {0: 'alt_allele',
                                               1: 'counted_allele'})
variant_table = pd.merge(variant_table,variant_last_field, on = variant_table.index)
column_names = {0:'chr',
                1:'chr_position',
                2:'ref_allele'}
variant_table = variant_table.rename(columns = column_names)
variant_table = variant_table.loc[:, ['chr',      'chr_position',
          'ref_allele','alt_allele',
       'counted_allele']]
variant_table['raw_column'] = variant_cols
variant_table['chr_position'] = variant_table['chr_position'].astype(int)

## 3. Prepare the score file (PGS002237)

In the PGS Catalog harmonized file, `hm_chr` / `hm_pos` are GRCh38 coordinates. A `chr` prefix is added (same format as `.raw`), rows with empty `hm_pos` and duplicated `(hm_chr, hm_pos, effect_allele)` are dropped, columns are renamed to `chr`, `chr_position`.

In [ ]:
def add_chr_prefix(x):
  if "chr" not in str(x):
    return(f"chr{x}")
  else:
    return(x)

beta_scores['chr_name'] = beta_scores['chr_name'].apply(lambda x: add_chr_prefix(x))

beta_scores['hm_chr'] = beta_scores['hm_chr'].apply(lambda x: add_chr_prefix(x))

In [ ]:
beta_scores.drop(beta_scores[beta_scores['hm_pos'].isnull()].index, inplace = True)

beta_scores = beta_scores.loc[beta_scores.loc[:, ['hm_chr','hm_pos','effect_allele']].drop_duplicates().index]

beta_scores['hm_pos'] = beta_scores['hm_pos'].astype(int)

beta_scores.drop(columns = ['chr_position'], inplace = True)

beta_scores.drop(columns = ['chr_name'], inplace = True)

beta_scores.rename(columns = {'hm_pos': 'chr_position',
                              'hm_chr': 'chr'}, inplace = True)

## 4. Match and orient the effect allele

`variant_table` (variants in the person) is merged with `beta_scores` on `chr` + `chr_position`; variants absent from the score file drop out.

Orientation rule:
* `alt_allele == effect_allele` → weight as is.
* `ref_allele == effect_allele` → weight `× (−1)` (the counted allele is the opposite of the effect allele).
* Neither (strand/allele mismatch) → the variant is in neither subset and **drops out**.

In [ ]:
matched_variants = pd.merge(left = variant_table, right =  beta_scores, on = ['chr_position', 'chr'])

In [ ]:
effect_is_alt = matched_variants[matched_variants['alt_allele'] == matched_variants['effect_allele']]
effect_is_ref = matched_variants[matched_variants['ref_allele'] == matched_variants['effect_allele']]
effect_is_ref['effect_weight'] = effect_is_ref['effect_weight']*(-1)

oriented_variants = pd.concat([effect_is_ref, effect_is_alt]).sort_index()

## 5. Dosage × weight → per-person PRS

The weight vector (`weight_row`, 1 row × variants) is replicated to the number of persons (`weight_matrix`), multiplied element-wise with the `.raw` dosage columns, and summed per row into `prs_raw`.

In [ ]:
weight_dict = {'variant':oriented_variants['raw_column'].values,
               'beta': oriented_variants['effect_weight'].values}

weight_row = pd.DataFrame(weight_dict).set_index("variant").T.reset_index(drop = True)

weight_matrix = pd.concat([weight_row, pd.DataFrame(columns = weight_row.columns, index = np.arange(0,1800)).drop(0)]).ffill()

dosage_raw.loc[:, weight_row.columns] = dosage_raw.loc[:, weight_row.columns] * weight_matrix

# dosage_raw.loc[:, weight_row.columns] * weight_row.iloc[0]

In [ ]:
dosage_raw['prs_raw'] = dosage_raw.loc[:, weight_row.columns].sum(axis = 1)

## 6. Sanity check: mean PRS by ancestry

`research_id` in `ancestry_preds` = `IID`. Mean PRS is expected to differ between ancestry groups (a PRS depends on its training population); this table is for review, not a result.

In [ ]:
ancestry_preds.rename(columns = {'research_id':"IID"}, inplace = True)

In [ ]:
ancestry_check = pd.merge(dosage_raw, ancestry_preds, on = 'IID').sort_values('ancestry_pred')

ancestry_check.groupby('ancestry_pred')["prs_raw"].mean()

In [ ]:
print("variant columns in .raw:", len(variant_table))
print("matched to score file:", len(matched_variants))
print("oriented (alt = effect):", len(effect_is_alt), "| (ref = effect, sign flipped):", len(effect_is_ref))
print("dropped (neither allele matched):", len(matched_variants) - len(oriented_variants))
print("persons (.raw):", len(dosage_raw), "| prs_raw NaN:", int(dosage_raw['prs_raw'].isna().sum()))
print("APOL1 persons before/after QC:", len(apol1_raw), "/", len(apol1_qc))
print(apol1_table['apol1_high_risk'].value_counts().to_dict())